In [1]:
import json
import numpy as np
import pandas as pd
import seaborn as plt
import matplotlib.pyplot as plt

In [2]:
def plot_global_analysis(json_path: str, exp_name: str, max_k: int = 50):
    # 1. 데이터 로드
    print(f"Loading data from {json_path}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"Global Schema Linking Performance: {exp_name}", fontsize=18, fontweight='bold', y=1.05)

    # ==========================================
    # Plot 1: Global Violin Plot (Separability)
    # ==========================================
    # 바이올린 플롯으로 두 클래스(Noise vs Gold)의 점수 분포와 밀도를 시각화
    sns.violinplot(data=df, x='is_gold', y='score', 
                   palette={True: '#1f77b4', False: '#d62728'}, 
                   inner='quartile', ax=axes[0])
    
    axes[0].set_title("Global Node Score Distribution", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Is Gold Schema? (True=Positive, False=Noise)", fontsize=12)
    axes[0].set_ylabel("Predicted Score (Cosine Similarity)", fontsize=12)
    axes[0].set_xticklabels(['False (Noise)', 'True (Gold)'])

    # ==========================================
    # Plot 2: Recall@K Curve (Retrieval Performance)
    # ==========================================
    print("Calculating Recall@K across all queries...")
    
    # 쿼리별로 그룹화하여 계산
    recall_at_k_list = {k: [] for k in range(1, max_k + 1)}
    
    grouped = df.groupby('query_id')
    for query_id, group in grouped:
        # 점수 내림차순 정렬
        sorted_group = group.sort_values(by='score', ascending=False).reset_index(drop=True)
        total_golds = sorted_group['is_gold'].sum()
        
        # Gold 노드가 아예 없는 예외적 쿼리는 스킵
        if total_golds == 0:
            continue
            
        # 1부터 max_k까지 Recall 계산
        for k in range(1, max_k + 1):
            hits_in_top_k = sorted_group.head(k)['is_gold'].sum()
            recall = hits_in_top_k / total_golds
            recall_at_k_list[k].append(recall)

    # 전체 쿼리에 대한 평균 Recall@K 산출
    k_values = list(range(1, max_k + 1))
    mean_recalls = [np.mean(recall_at_k_list[k]) * 100 for k in k_values]

    # 라인 플롯 그리기
    axes[1].plot(k_values, mean_recalls, marker='o', markersize=4, linestyle='-', color='#2ca02c', linewidth=2)
    axes[1].axhline(y=100, color='gray', linestyle='--', alpha=0.7)
    
    axes[1].set_title(f"Average Recall@K Curve (Up to Top-{max_k})", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("K (Number of Retrieved Nodes)", fontsize=12)
    axes[1].set_ylabel("Recall (%)", fontsize=12)
    axes[1].set_ylim(0, 105)
    axes[1].grid(True, linestyle=':', alpha=0.6)

    # 텍스트 주석 (Top-10, Top-20 등 주요 지점 수치 표기)
    for k_idx in [5, 10, 20, 50]:
        if k_idx <= max_k:
            axes[1].annotate(f"{mean_recalls[k_idx-1]:.1f}%", 
                             (k_idx, mean_recalls[k_idx-1]), 
                             textcoords="offset points", xytext=(0,10), ha='center', fontsize=10)

    plt.tight_layout()
    # save_path = f"global_performance_{exp_name}.png"
    # plt.savefig(save_path, dpi=300, bbox_inches='tight')
    # print(f"📊 Visualization saved to: {save_path}")
    plt.show()

# 1. Preliminary: Vector Only

In [ ]:
json_path = ""
exp_name= ""
max_k = 50


plot_global_analysis(json_path, exp_name, max_k)